# Derivation of EWMA Variance Estimate's Recursive Formula
Define the EWMA variance estimate at time $t$ as an exponentially weighted average of past squared returns:
$$σ^2_t = (1 - λ)\sum_{i=1}^{\infty}λ^{i - 1}r^2_{t - i}\quad(1)$$

where $r_{t-i}$ is the log return $i$ periods before time $t$, $λ \in (0, 1)$ is the decay factor (RiskMetrics uses $λ = 0.94$ for daily data), and $(1 - λ)$ is a normalizing constant so the weights sum to 1.

Now consider the one-step back formula,
$$σ^2_{t - 1} = (1 - λ)\sum_{i=1}^{\infty}λ^{i - 1}r^2_{t - 1 - i}\quad(2)$$

In (1), we can isolate the first term in the sum:
$$σ^2_t = (1 - λ)r^2_{t - 1} + (1 - λ)\sum_{i=2}^{\infty}λ^{i - 1}r^2_{t - i}$$

And then reindex the sum:
$$σ^2_t = (1 - λ)r^2_{t - 1} + (1 - λ)\sum_{i=1}^{\infty}λ^{i}r^2_{t - 1 - i}$$

And then factor out a $λ$ from the sum:
$$σ^2_t = (1 - λ)r^2_{t - 1} + λ(1 - λ)\sum_{i=1}^{\infty}λ^{i - 1}r^2_{t - 1 - i}$$

Notice the sum and the $(1 - λ)$ factor are now just $σ^2_{t - 1}$ from (2):
$$σ^2_t = λσ^2_{t - 1} + (1 - λ)r^2_{t - 1}\quad(3)$$

Which gives our desired final recursive formula (cross referenced with the fourth edition of "J.P. Morgan/Reuters, RiskMetrics — Technical Document, New York", p. 78-82, hosted on MSCI).

# EWMA's Relation To GARCH (Optional)
The EWMA model is a special, restricted case of the standard GARCH(1,1) model where the constant term is zero and the persistence weights sum to one, i.e. it is a simpler case.

The GARCH(1, 1) equation, without derivation, is as follows:
$$σ^2_t = \omega + {\alpha}r^2_{t - 1} + \betaσ^2_{t - 1}\quad(4)$$

In EWMA, $\omega = 0$ and $\alpha + \beta = 1$, unlike GARCH(1, 1) where it is required that $\omega > 0$ and $\alpha + \beta < 1$. In addition, GARCH(1, 1) allows our model to exhibit mean reversion, i.e. a pull of volatility towards a "long-run unconditional variance level", while EWMA simply assumes volatility follows a random walk. In terms of estimation, EWMA is much simpler (**and thus why it's the choice in the first major version of this project**) as it doesn't require complex parameter optimization because λ is chosen subjectively or preset by convention (e.g., 0.94 in RiskMetrics). On the other hand, GARCH(1, 1) requires statistical estimation (MLE) to fit all the parameters to the sample reference.

# Derivation of Parametric Equations for VaR and ES
## Preliminary Calculations and Assumptions
**Assumptions**: 
- We denote the confidence level as $\alpha$. 
- We use the CDF definition for critical z-values, i.e. $\mathbb{P}(Z < z_\alpha) = \alpha$ or $\Phi(z_\alpha) = \alpha$. 
- We assume prices/portfolio values (since in our case they are the same) $P$ to be lognormally distributed, and log returns $R$ to be normally distributed s.t. $R \sim \mathcal{N}(\mu_R, \sigma_R^2)$ where $μ_R = Δt(μ - \frac{1}{2}σ^2)$ and $σ_R^2 = Δtσ^2$. 
- In our code, $σ^2$ will be $σ_{T+1}^2$, i.e. the forecasted variance.
- We assume GBM holds.

**Preliminary Calculations**:
Let the terminal loss L at time horizon T be $L = P_0 - P_T$ where $P_T$, by GBM, is
$$P_T = P_0e^{Δt(μ - \frac{1}{2}σ^2) + \sqrt{Δt}σW} = P_0e^R$$

For small returns, we can use a first-order Taylor approximation of the exponential function to approximate $P_T$ for small R, which is the case here given we are dealing with small daily log returns. Thus,
$$P_T \approx P_0(1 + R)$$

Now rewrite, $L = P_0 - P_T$:
$$L = P_0 - P_T \approx P_0 - P_0(1 + R) = -P_0R\quad(1)$$

## Parametric Equation for Value at Risk
We begin by writing the probability that the loss exceeds the VaR (derived from the def. of VaR: $VaR_\alpha = inf\{x : \mathbb{P}(L \leq x) \geq \alpha\}$):
$$\mathbb{P}(L > VaR_\alpha) = 1 - \alpha$$

Substituting (1) into the above probability gives
$$\mathbb{P}(-P_0R > VaR_\alpha) = 1 - \alpha$$

Since $P_0$ is positive, dividing both sides of the inequality $−P_0R > VaR_α$ by $−P_0$ (since $P_0 > 0$) reverses the inequality:
$$\mathbb{P}(R < -\frac{VaR_\alpha}{P_0}) = 1 - \alpha$$

Given that $R$ is normally distributed, we can standardize it using the z-score transformation:

$$Z = \frac{R - μ_R}{σ_R}$$

This allows the probability statement to be rewritten in terms of the standard normal distribution:
$$\mathbb{P}(Z < \frac{\frac{-VaR_\alpha}{P_0} - μ_R}{σ_R}) = 1 - \alpha$$

Since $z_{1-α}$ is the z-score corresponding to the lower-tail probability of $1 - α$, the equation becomes
$$\frac{-(\frac{VaR_\alpha}{P_0} + μ_R)}{σ_R} = z_{1 - \alpha}$$

Due to the symmetric properties of standard normal distributions, we know $z_{1 - \alpha} = -z_{\alpha}$. Thus,
$$\frac{-(\frac{VaR_\alpha}{P_0} + μ_R)}{σ_R} = -z_{\alpha}$$

We proceed with the simplification to solve for $VaR_{\alpha}$:
$$\frac{\frac{VaR_\alpha}{P_0} + μ_R}{σ_R} = z_{\alpha}$$

$$\frac{VaR_\alpha}{P_0} + μ_R = σ_Rz_{\alpha}$$

$$\frac{VaR_\alpha}{P_0} = σ_Rz_{\alpha} - μ_R $$

$$VaR_\alpha = -P_0\left(μ_R - σ_Rz_{\alpha}\right) = -P_0\left[Δt\left(μ - \frac{1}{2}σ^2\right) - \sqrt{Δt}σz_{\alpha}\right]\quad(2)$$

## Parametric Equation for Expected Shortfall/Conditional Value at Risk
We begin by writing the definition of ES at confidence level $\alpha$:
$$ES_{\alpha} = \mathbb{E}\left[L \;|\;  L > VaR_{\alpha}\right]$$

Substituing $L \approx -P_0R$ and (2):
$$ES_{\alpha} = \mathbb{E}\left[-P_0R \;|\;  -P_0R > -P_0\left(μ_R - σ_Rz_{\alpha}\right)\right]$$

Cancelling out $-P_0$ from the inequality (this requires flipping the inequality sign) and using linearity of expectation:
$$ES_{\alpha} = -P_0\mathbb{E}\left[R \;|\;  R < μ_R - σ_Rz_{\alpha}\right]$$

Since R is normally distributed, we can substitute with a z-score transformation as we did in the VaR calculations:
$$ES_{\alpha} = -P_0\mathbb{E}\left[μ_R + σ_Rz \;|\;  μ_R + σ_Rz < μ_R - σ_Rz_{\alpha}\right]$$

By linearity of expectation,
$$ES_{\alpha} = -P_0\left(μ_R + σ_R\mathbb{E}\left[z \;|\;  μ_R + σ_Rz < μ_R - σ_Rz_{\alpha}\right]\right)$$

We now simplify the inequality:
$$ES_{\alpha} = -P_0\left(μ_R + σ_R\mathbb{E}\left[z \;|\;  z < -z_{\alpha}\right]\right)\quad(*)$$

We take a detour to calculate $\mathbb{E}[z \;|\;  z < -z_{\alpha}]$:
$$\mathbb{E}[z \;|\;  z < -z_{\alpha}] = \int_{z_{\alpha}}^{\infty}z\varphi_{z \,|\, z < -z_{\alpha}(z)}\;dz$$

where $\varphi(z)$ is the PDF of a standard normal distribution and $\varphi_{z \,|\, z < -z_{\alpha}(z)}$ its conditional PDF s.t.
$$\varphi_{z \,|\, z < -z_{\alpha}(z)} = \frac{\varphi(z)}{\mathbb{P}(z < -z_{\alpha})}$$

by def. of conditional PDFs. Thus, subsituting the above and $\mathbb{P}(z < -z_{\alpha}) = \mathbb{P}(z < z_{1 - \alpha}) = 1 - \alpha$:
$$\mathbb{E}\left[z \;|\;  z < -z_{\alpha}\right] = \frac{1}{1 - \alpha} \int_{-\infty}^{-z_{\alpha}}z\varphi(z)\;dz$$

Reminder: the PDF of a standard normal r.v. is
$$\varphi(z) = \frac{1}{\sqrt{2\pi}}e^{\frac{-z^2}{2}}$$

Thus, subsituting again
$$\mathbb{E}\left[z \;|\;  z < -z_{\alpha}\right] = \frac{1}{1 - \alpha} \int_{-\infty}^{-z_{\alpha}}\frac{z}{\sqrt{2\pi}}e^{\frac{-z^2}{2}}\;dz$$

Let $u = \frac{-z^2}{2} \implies du = -zdz, u_{lower} \to -\infty, u_{upper} = \frac{-z_{\alpha}^2}{2}$ and subsitute
$$\mathbb{E}\left[z \;|\;  z < -z_{\alpha}\right] = \frac{1}{\sqrt{2\pi}(1 - \alpha)} \int_{-\infty}^{\frac{-z_{\alpha}^2}{2}}-e^{u}\;du$$

Compute the integral:
$$\mathbb{E}\left[z \;|\;  z < -z_{\alpha}\right] = \frac{-1}{\sqrt{2\pi}(1 - \alpha)} \lim_{t \to -\infty}e^{u}\vert_t^{\frac{-z_{\alpha}^2}{2}}$$

We now evaluate:
$$\mathbb{E}\left[z \;|\;  z < -z_{\alpha}\right] = \frac{-1}{1 - \alpha}\left(\frac{1}{\sqrt{2\pi}} e^{\frac{-z_{\alpha}^2}{2}}\right)$$

We see the term in the parentheses is simply the PDF of a standard normal r.v. evaluated at $z_{\alpha}$. Thus,
$$\mathbb{E}\left[z \;|\;  z < -z_{\alpha}\right] = \frac{-\varphi(z_{\alpha})}{1 - \alpha}$$

Now, we'll use the above in (*)
$$ES_{\alpha} = -P_0\left(μ_R - \frac{σ_R\varphi(z_{\alpha})}{1 - \alpha}\right)$$

Now we simplify and make final subsitutions with $μ_R = Δt(μ - \frac{1}{2}σ^2)$, $σ_R = \sqrt{Δt}σ$ and the PDF of a standard normal r.v.:
$$ES_{\alpha} = -P_0\left[Δt\left(μ - \frac{1}{2}σ^2\right) - \frac{\sqrt{Δt}σe^{\frac{-z_{\alpha}^2}{2}}}{\sqrt{2\pi}(1 - \alpha)}\right]\quad(3)$$

We now have two parametric (closed-form) equations for VaR and ES. Interestingly, they are very similar, with the only difference being in the "standard deviation" terms, which is especially apparent if we let $z_\alpha^* = \frac{\varphi(z_{\alpha})}{1 - \alpha}$ and then get:

$$\left\{ \begin{aligned}
  VaR_\alpha &= -P_0\left[Δt\left(μ - \frac{1}{2}σ^2\right) - \sqrt{Δt}σz_{\alpha}\right] \\
  ES_{\alpha} &= -P_0\left[Δt\left(μ - \frac{1}{2}σ^2\right) - \sqrt{Δt}σz_{\alpha}^*\right]
\end{aligned} \right.$$

which really shows how similar these parametric equations are.

In [409]:
# for modules to work when running this notebook, we need to add the project root to the sys.path
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [410]:
#imports
import numpy as np
import pandas as pd
import numpy.typing as npt
import scipy.stats as stats

from data.data import get_multiple_stocks_data

In [411]:
# reuseable class for GBM simulation and risk metrics calculation
class EWMAEngine:
    ROLLING_PERIODS = (20, 60)
    TIME_HORIZONS = (1, 10)
    ALPHAS = (0.95, 0.99) # the confidence levels for VaR and ES calculations
    DAILY = 252
    N_PATHS = 10_000
    LAMBDA_ = 0.94
    T = 1.0  # Time to maturity in years for the simulation
    N = 252  # Number of time steps in the simulation (daily steps for 1 year)

    def get_conf(self, alpha: float) -> int:
        if alpha not in self.ALPHAS:
            raise ValueError(f"Alpha must be one of {self.ALPHAS}")
        return int(alpha * 100)

    def ewma_volatility(self, log_returns: pd.Series, lambda_: float = LAMBDA_) -> float:
        if (not 0 < lambda_ <= 1):
            raise ValueError("Lambda must be between 0 and 1")

        squared_returns = log_returns ** 2
        last_return = squared_returns.iloc[-1]
        squared_returns = squared_returns.shift(1)
        squared_returns = squared_returns.fillna(value=log_returns.mean() ** 2)
        ewma_var = squared_returns.ewm(alpha=1 - lambda_, adjust=False).mean().iloc[-1] 

        forecast = (last_return * (1 - lambda_) + ewma_var  * lambda_) ** 0.5 * np.sqrt(self.DAILY)
        
        return forecast

    def simulate_gbm(
        self,
        S0: float,
        mu_annual: float,
        sigma_annual: float,
        T: float,
        N: int,
        n_paths: int
    ) -> npt.NDArray[np.float64]:
        N = int(N)
        dt = T / N
        z = np.random.standard_normal((n_paths, N))

        log_increments = (
            (mu_annual - 0.5 * sigma_annual**2) * dt
            + sigma_annual * np.sqrt(dt) * z
        )

        paths = np.zeros((n_paths, N + 1))
        paths[:, 0] = S0
        paths[:, 1:] = S0 * np.exp(np.cumsum(log_increments, axis=1))
        return paths

    def var_es(
        self,
        paths: npt.NDArray[np.float64],
        time_horizon: int,
        alpha: float
    ) -> tuple[float, float]:
        if time_horizon < 1 or time_horizon >= paths.shape[1]:
            raise ValueError("time_horizon must be between 1 and number of simulated steps")

        P0 = paths[0, 0]
        losses = P0 - paths[:, time_horizon]

        var = np.percentile(losses, alpha * 100)
        es = losses[losses > var].mean()

        return float(var), float(es)

    def var_es_parametric(
        self,
        paths: npt.NDArray[np.float64],
        mu: float,
        sigma: float,
        T: float,
        N: int,
        time_horizon: int,
        alpha: float
    ) -> tuple[float, float]:
        P0 = paths[0, 0]
        delta_t = (T / N) * time_horizon
        z_alpha = stats.norm.ppf(alpha)
        adjusted_mu = delta_t * (mu - 0.5 * sigma ** 2)
        adjusted_sigma = sigma * np.sqrt(delta_t)
        z_star_alpha = stats.norm.pdf(z_alpha) / (1 - alpha)

        var = -P0 * (adjusted_mu - adjusted_sigma * z_alpha)
        es = -P0 * (adjusted_mu - adjusted_sigma * z_star_alpha)

        return float(var), float(es)

In [412]:
# initialize the risk engine and variables to store data
ewma_eng = EWMAEngine()

tickers = ["SPY", "GLD", "NVDA", "GOOGL", "BTC-USD"]
data = get_multiple_stocks_data(tickers, "2016-06-01", "2026-06-01", "1d")

prices = {}
log_returns = {}
paths_by_ticker = {}
results = {}

In [413]:
# populate the dictionaries with prices, log returns, simulated paths, and risk metrics
for ticker in tickers:
    curr_data = data[ticker]
    prices[ticker] = curr_data["Close"]

    lr = pd.Series(
        np.diff(np.log(prices[ticker].values)),
        index=prices[ticker].index[1:]
    )
    log_returns[ticker] = lr

    mu_annual = lr.mean() * ewma_eng.DAILY 

    sigma_annual = ewma_eng.ewma_volatility(log_returns=lr)

    paths = ewma_eng.simulate_gbm(
        S0=prices[ticker].iloc[-1],
        mu_annual=mu_annual,
        sigma_annual=sigma_annual,
        T=1.0,
        N=ewma_eng.DAILY,
        n_paths=ewma_eng.N_PATHS
    )

    paths_by_ticker[ticker] = paths
    solving_methods = ["monte_carlo", "parametric"]
    results[ticker] = {method: {} for method in solving_methods}
         
    for horizon in ewma_eng.TIME_HORIZONS:
            horizon_key = f"{horizon}d"
            for method in solving_methods:
                results[ticker][method][horizon_key] = {}
    
            for alpha in ewma_eng.ALPHAS:
                var, es = ewma_eng.var_es(paths, horizon, alpha)
                para_var, para_es = ewma_eng.var_es_parametric(paths, mu_annual, sigma_annual, ewma_eng.T, ewma_eng.N, horizon, alpha)
                conf = ewma_eng.get_conf(alpha)

                results[ticker]["monte_carlo"][horizon_key][f"var_{conf}"] = var
                results[ticker]["monte_carlo"][horizon_key][f"es_{conf}"] = es
                results[ticker]["parametric"][horizon_key][f"var_{conf}"] = para_var
                results[ticker]["parametric"][horizon_key][f"es_{conf}"] = para_es
    

In [414]:
# ------------------------------------------------------------------
# 1. Flatten the nested `results` dict into a tidy long-format table
# ------------------------------------------------------------------
metric_labels = {
    "var_95": "VaR 95%",
    "es_95": "ES 95%",
    "var_99": "VaR 99%",
    "es_99": "ES 99%",
}
metrics_order = ["var_95", "es_95", "var_99", "es_99"]

rows = []
for ticker in tickers:
    for horizon_key in results[ticker]["monte_carlo"]:
        for metric in metrics_order:
            mc_val = results[ticker]["monte_carlo"][horizon_key][metric]
            para_val = results[ticker]["parametric"][horizon_key][metric]
            rows.append({
                "Ticker": ticker,
                "Horizon": horizon_key,
                "Metric": metric_labels[metric],
                "Monte Carlo": mc_val,
                "Parametric": para_val,
                "Difference (MC - CF)": mc_val - para_val,
            })

comparison_table = pd.DataFrame(rows)

# ------------------------------------------------------------------
# 2. Enforce sensible ordering (numeric horizon, fixed metric order)
# ------------------------------------------------------------------
horizon_order = sorted(
    comparison_table["Horizon"].unique(),
    key=lambda x: int(x.replace("d", ""))
)
metric_order = [metric_labels[m] for m in metrics_order]

comparison_table["Horizon"] = pd.Categorical(
    comparison_table["Horizon"], categories=horizon_order, ordered=True
)
comparison_table["Metric"] = pd.Categorical(
    comparison_table["Metric"], categories=metric_order, ordered=True
)

comparison_table = comparison_table.sort_values(
    ["Ticker", "Horizon", "Metric"]
).reset_index(drop=True)

comparison_grouped = comparison_table.set_index(["Ticker", "Horizon", "Metric"])

# ------------------------------------------------------------------
# 3. Detect group boundaries so we know where to draw borders
# ------------------------------------------------------------------
idx = comparison_grouped.index
ticker_vals = idx.get_level_values("Ticker").to_numpy()
horizon_vals = idx.get_level_values("Horizon").to_numpy()
n = len(idx)

new_ticker = np.zeros(n, dtype=bool)   # True where a new Ticker group starts
new_horizon = np.zeros(n, dtype=bool)  # True where a new Horizon sub-group starts
for i in range(1, n):
    new_ticker[i] = ticker_vals[i] != ticker_vals[i - 1]
    new_horizon[i] = new_ticker[i] or (horizon_vals[i] != horizon_vals[i - 1])

TICKER_BORDER = "border-top: 3px solid #4695f0;"   # thick dark line between tickers
HORIZON_BORDER = "border-top: 1px solid #b0b7bd;"  # thin gray line between horizons

def body_border_func():
    """Applies the right border to every data cell in a row, based on group boundaries."""
    counter = {"i": 0}
    def func(row):
        i = counter["i"]
        counter["i"] += 1
        css = TICKER_BORDER if new_ticker[i] else (HORIZON_BORDER if new_horizon[i] else "")
        return [css] * len(row)
    return func

def index_border_func(level_changed_array, border_css):
    """Same border logic, applied to the sparse index cells (Ticker/Horizon columns)."""
    def func(s):
        return [border_css if level_changed_array[i] else "" for i in range(len(s))]
    return func

# ------------------------------------------------------------------
# 4. Style: number formatting, diff color gradient, header, borders,
#    and a hover effect that also darkens the gradient-colored cells
# ------------------------------------------------------------------
CMAP = "coolwarm"

diff_vals = comparison_grouped["Difference (MC - CF)"]

styled = (
    comparison_grouped.style
    .format({
        "Monte Carlo": "{:.4f}",
        "Parametric": "{:.4f}",
        "Difference (MC - CF)": "{:+.4f}",
    })
    .background_gradient(
            subset=["Difference (MC - CF)"],
            cmap=CMAP,          # note: NOT coolwarm_r, see below
            gmap=diff_vals.abs(),     # color driven by |difference|, not the signed value
            vmin=0,                  # zero deviation -> low end of cmap
            vmax=diff_vals.abs().max()  # biggest deviation -> high end
        )
    .set_properties(**{"text-align": "center"})
    .apply(body_border_func(), axis=1)
    .apply_index(index_border_func(new_horizon, HORIZON_BORDER), axis=0, level=1)
    .apply_index(index_border_func(new_horizon, HORIZON_BORDER), axis=0, level=2)
    .apply_index(index_border_func(new_ticker, TICKER_BORDER), axis=0, level=0)
    .apply_index(index_border_func(new_ticker, TICKER_BORDER), axis=0, level=1)
    .apply_index(index_border_func(new_ticker, TICKER_BORDER), axis=0, level=2)
    .set_table_styles([
        {"selector": "th", "props": [
            ("text-align", "center"),
            ("background-color", "#2c3e50"),
            ("color", "white"),
            ("font-weight", "bold"),
            ("padding", "6px 10px"),
        ]},
        {"selector": "td", "props": [("padding", "6px 10px")]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("font-family", "Arial, sans-serif"),
            ("font-size", "13px"),
        ]},
        # Hover for plain (non-gradient) cells: simple background tint.
        {"selector": "tbody tr", "props": [("transition", "background-color 0.3s ease")]},
        {"selector": "tbody tr:hover", "props": [("background-color", "#494a4b")]},
        # Hover overlay for gradient cells: an inset box-shadow layers a
        # translucent dark tint ON TOP of the inline background-color that
        # background_gradient() sets, since background-color itself can't
        # be reliably overridden by a plain CSS hover rule once it's inline.
        {"selector": "tbody tr td", "props": [("transition", "box-shadow 0.3s ease")]},
        {"selector": "tbody tr:hover td", "props": [
            ("box-shadow", "inset 0 0 0 9999px rgba(0,0,0,0.12)")
        ]},
        {"selector": "caption", "props": [
                    ("font-size", "14px"),
                    ("font-weight", "bold"),
                    ("padding", "8px 0"),
                ]},
    ], overwrite=False)  # overwrite=False keeps this appended, not replacing prior rules
    .set_caption("VaR / ES Comparison — Monte Carlo vs Parametric/Closed-Form")
)

styled # show table in notebook

# Model limitations
We continue with the assumptions/limitations mentioned in Notebook 02, which I will transcribe verbatim here aswell: 

- Under GBM, prices are lognormally distributed and log returns are normally distributed with iid increments. This is the distributional assumption behind the Monte Carlo VaR baseline, but Notebook 01 shows that real market returns are not normal.
- GBM assumes continuous price paths, so it cannot capture jumps or discontinuities.
- GBM assumes constant volatility, while real markets exhibit time-varying volatility, clustering, and, for many assets, skew/smile effects.

In addition, with the addition of a volatility forecast calculated under a EWMA model and the parametric equations, we have new assumptions/limitations:

- The log returns obtained from simulations are small enough that we can use a first-order Taylor approximation for $e$.
- EWMA doesn't have mean reversion properties that GARCH(1, 1) would have, but instead we assume volatility to be a random walk.

# Short interpretation of results
From the simulation, we see that the risk metrics deduced from the Monto Carlo method are exclusively/almost exclusively underestimating VaR and ES compared to the values obtained from the closed-form equations, by generally small margins (close to 0, under orders of tens) for lower volatility tickers. On the other hand, we see a breach of our assumptions when we take into account higher volatility assets like BTC, where the difference shoots up to orders of hundreds. This massive jump is most likely due to our first-order Taylor approximation of $e$ used in the derivation of the equations, as it requies (log) returns to be small, while returns for high volatility assets can easily surpass that arbitrary "small" threshold.